# sestudio on Google Colab

1. **Runtime → Change runtime type → T4 GPU** (once; AI frames need it).
2. Press ▶ on the cell below and allow access to Google Drive.
3. Open the big link it shows — from any device, no sign-in. Share it only with people you trust: whoever has it controls sestudio.

Settings, your library and downloaded videos are kept in the Drive folder below, so the next session picks up where this one stopped. Press ■ to stop; it saves first.

In [ ]:
#@title ▶ Start sestudio
drive_folder = "sestudio"  #@param {type:"string"}

import json, os, re, shutil, sqlite3, socket, subprocess, time, urllib.request
from pathlib import Path
from google.colab import drive
from IPython.display import HTML, display

drive.mount("/content/drive")
saved = Path("/content/drive/MyDrive") / drive_folder
(saved / "videos").mkdir(parents=True, exist_ok=True)

# Settings and library run from local disk: SQLite on the Drive mount risks
# corruption, and the app's caches (GBs of GPU runtime) would crawl there.
local = Path("/content/sestudio")
local.mkdir(exist_ok=True)
for name in ("config.json", "library.db"):
    if (saved / name).exists():
        shutil.copy(saved / name, local / name)
config_path = local / "config.json"
config = json.loads(config_path.read_text()) if config_path.exists() else {}
config["output_root"] = str(saved / "videos")
config_path.write_text(json.dumps(config, indent=2))

def save():
    """Copy settings and library to Drive; the backup API is safe while the app writes."""
    shutil.copy(config_path, saved / "config.json")
    if (local / "library.db").exists():
        src, dst = sqlite3.connect(local / "library.db"), sqlite3.connect(saved / "library.db")
        src.backup(dst)
        src.close(); dst.close()

def port_open(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

print("Installing and starting sestudio (about a minute the first time)…")
subprocess.run(["pip", "install", "-q", "uv"], check=True)
env = {**os.environ, "SESTUDIO_CONFIG": str(config_path), "SESTUDIO_DB": str(local / "library.db")}
server = subprocess.Popen(
    ["uvx", "sestudio", "--host", "127.0.0.1", "--http_port", "8080", "--https_port", "8443"],
    env=env, stdout=open(local / "server.log", "w"), stderr=subprocess.STDOUT,
)
while not (port_open(8080) and port_open(8443)):
    if server.poll() is not None:
        raise SystemExit("sestudio did not start:\n" + (local / "server.log").read_text()[-3000:])
    time.sleep(2)

cloudflared = local / "cloudflared"
if not cloudflared.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)
tunnel_log = local / "tunnel.log"
tunnel = subprocess.Popen(
    # --no-tls-verify: sestudio's certificate is self-signed; the public side gets Cloudflare's.
    [str(cloudflared), "tunnel", "--url", "https://127.0.0.1:8443", "--no-tls-verify", "--no-autoupdate"],
    stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT,
)
url = None
for _ in range(60):
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text())
    if found:
        url = found.group(0)
        break
    time.sleep(1)
if url is None:
    raise SystemExit("The public link could not be created:\n" + tunnel_log.read_text()[-3000:])

display(HTML(f"""
<div style="font-family:sans-serif;padding:24px;border-radius:12px;background:#111;color:#eee;text-align:center">
  <div style="font-size:15px;opacity:.8">sestudio is running. Open it on any device:</div>
  <a href="{url}" target="_blank" style="display:inline-block;margin:16px 0;padding:16px 28px;border-radius:10px;
     background:#3b82f6;color:#fff;font-size:22px;font-weight:600;text-decoration:none">Open sestudio</a>
  <div style="font-size:18px;user-select:all">{url}</div>
  <div style="font-size:13px;opacity:.6;margin-top:12px">The link may take up to a minute to answer. It changes every session.
  Keep this tab open; press ■ to stop.</div>
</div>"""))

try:
    while True:
        time.sleep(300)
        save()
        # Not server.poll(): a restart from Settings replaces the process, the port stays.
        if not port_open(8443):
            time.sleep(30)
            if not port_open(8443):
                print("sestudio stopped:\n" + (local / "server.log").read_text()[-3000:])
                break
finally:
    save()
    tunnel.terminate()
    server.terminate()
    print("Saved to Drive. Stopped.")
